# Random Forest baseline
Train a fresh Random Forest with identical settings on each chronological split and predict the probability of illicit activity (`label = 1`). Use only the 165 supplied features and known labels.

The split boundaries live in `validation_splits.json`, also read by notebook 1. No separate CSV copies of validation data are needed. This notebook runs independently of the inspection notebooks.

Both periods are evaluated here. If you use both scores to select models or settings, both are development validation folds: the name “latest holdout” does **not** make its score an untouched final evaluation.

Required packages: pandas, numpy, scikit-learn, and a Python Jupyter kernel.

In [1]:
from pathlib import Path
import json
import time
import platform
import numpy as np
import pandas as pd
import sklearn
from IPython.display import display
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    roc_auc_score, average_precision_score, precision_score,
    recall_score, f1_score, confusion_matrix,
)

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "bt-4012-competition-2026").is_dir()), None)
if ROOT is None:
    raise FileNotFoundError("Open this notebook inside the competition project.")
DATA = ROOT / "bt-4012-competition-2026"
splits = json.loads((ROOT / "validation_splits.json").read_text())
FEATURES = [f"feat_{i}" for i in range(1, 166)]
SEED = 42
THRESHOLD = 0.5
MODEL_PARAMS = dict(
    n_estimators=200, max_depth=None, min_samples_leaf=1,
    max_features="sqrt", class_weight=None, random_state=SEED, n_jobs=2,
)
print("Python:", platform.python_version(), "scikit-learn:", sklearn.__version__)
display(pd.DataFrame(splits))

Python: 3.13.1 scikit-learn: 1.7.1


,name,train_start,train_end,validation_start,validation_end
0,earlier check,1,21,22,28
1,latest holdout,1,28,29,35


## Load known labels and check inputs
Unknown labels cannot supervise a classifier. No feature imputation is needed if the completeness check passes. Trees do not require feature scaling. `txId` is retained for overlap checks; it is excluded from the predictors, along with `time_step`.

The initial baseline uses no class weighting or resampling. These can be separate experiments later.

In [2]:
data = pd.read_csv(DATA / "train.csv", usecols=["txId", "time_step", *FEATURES, "label"])
unexpected = data.loc[data.label.notna() & ~data.label.isin([0, 1]), "label"].unique()
if len(unexpected):
    raise ValueError(f"Unexpected labels: {unexpected}")
labelled = data.loc[data.label.isin([0, 1])].copy()
labelled["label"] = labelled["label"].astype(int)
assert labelled.txId.is_unique, "Duplicate transaction IDs need investigation."
assert labelled.time_step.notna().all(), "Missing time steps need investigation."
assert np.isfinite(labelled[FEATURES].to_numpy()).all(), "Missing/infinite features need investigation."
print(f"Known labels available: {len(labelled):,}")
print(f"Unknown labels excluded: {data.label.isna().sum():,}")
print(f"Predictor columns: {len(FEATURES)}")
display(labelled.label.value_counts().rename_axis("label").to_frame("count"))

Known labels available: 31,235
Unknown labels excluded: 110,537
Predictor columns: 165


,count
label,
0,27591
1,3644


## Fit and evaluate both chronological folds
Each loop iteration creates a new model. ROC-AUC and average precision use probabilities, without a classification threshold. Precision, recall, F1, and the confusion matrix use a fixed threshold of 0.5; this threshold has not been tuned.

Average precision summarizes the precision–recall curve (it is not the trapezoidal PR-AUC). The training ROC-AUC is a diagnostic for memorization, not a generalization score. No validation labels or validation-fitted transformations enter training.

In [3]:
results = []
confusions = {}
for split in splits:
    train_fold = labelled.loc[labelled.time_step.between(split["train_start"], split["train_end"])]
    val_fold = labelled.loc[labelled.time_step.between(split["validation_start"], split["validation_end"])]
    assert split["train_end"] < split["validation_start"], "Time windows overlap."
    assert not set(train_fold.txId) & set(val_fold.txId), "Transaction overlap detected."
    assert train_fold.label.nunique() == val_fold.label.nunique() == 2, "Both classes are required."
    X_train, y_train = train_fold[FEATURES], train_fold.label
    X_val, y_val = val_fold[FEATURES], val_fold.label

    model = RandomForestClassifier(**MODEL_PARAMS)
    started = time.perf_counter()
    model.fit(X_train, y_train)
    fit_seconds = time.perf_counter() - started
    illicit_column = list(model.classes_).index(1)
    probabilities = model.predict_proba(X_val)[:, illicit_column]
    train_probabilities = model.predict_proba(X_train)[:, illicit_column]
    predictions = (probabilities >= THRESHOLD).astype(int)

    results.append({
        "split": split["name"], "train_steps": f'{split["train_start"]}–{split["train_end"]}',
        "validation_steps": f'{split["validation_start"]}–{split["validation_end"]}',
        "train_rows": len(train_fold), "validation_rows": len(val_fold),
        "validation_illicit": int(y_val.sum()), "validation_illicit_share": float(y_val.mean()),
        "train_roc_auc": roc_auc_score(y_train, train_probabilities),
        "validation_roc_auc": roc_auc_score(y_val, probabilities),
        "average_precision": average_precision_score(y_val, probabilities),
        "threshold": THRESHOLD,
        "illicit_precision": precision_score(y_val, predictions, zero_division=0),
        "illicit_recall": recall_score(y_val, predictions, zero_division=0),
        "illicit_f1": f1_score(y_val, predictions, zero_division=0),
        "fit_seconds": fit_seconds,
    })
    confusions[split["name"]] = pd.DataFrame(
        confusion_matrix(y_val, predictions, labels=[0, 1]),
        index=["Actual licit", "Actual illicit"],
        columns=["Predicted licit", "Predicted illicit"],
    )
    print(f'{split["name"]}: ROC-AUC = {results[-1]["validation_roc_auc"]:.4f}; '
          f'fit time = {fit_seconds:.1f}s', flush=True)

results_df = pd.DataFrame(results)
with pd.option_context("display.max_columns", None):
    display(results_df.round(4))
for name, matrix in confusions.items():
    print(name)
    display(matrix)

earlier check: ROC-AUC = 0.8765; fit time = 4.0s


latest holdout: ROC-AUC = 0.9950; fit time = 5.3s


,split,train_steps,validation_steps,train_rows,validation_rows,validation_illicit,validation_illicit_share,train_roc_auc,validation_roc_auc,average_precision,threshold,illicit_precision,illicit_recall,illicit_f1,fit_seconds
0,earlier check,1–21,22–28,19530,5677,671,0.1182,1.0,0.8765,0.8469,0.5,0.9754,0.7675,0.8590,3.9867
1,latest holdout,1–28,29–35,25207,6028,1102,0.1828,1.0,0.9950,0.9855,0.5,0.9952,0.9319,0.9625,5.3351


earlier check


,Predicted licit,Predicted illicit
Actual licit,4993,13
Actual illicit,156,515


latest holdout


,Predicted licit,Predicted illicit
Actual licit,4921,5
Actual illicit,75,1027


## Compare folds and save the experiment
Inspect each fold's result before the mean: performance may differ across periods. The mean below weights the two folds equally. These two scores are not a confidence interval.

The saved summary and settings allow later feature or model experiments to use the same reference. Rerunning this notebook replaces this baseline's result files.

In [4]:
metrics = ["validation_roc_auc", "average_precision", "illicit_precision", "illicit_recall", "illicit_f1"]
display(results_df.set_index("split")[metrics].round(4))
print(f'Mean fold ROC-AUC: {results_df.validation_roc_auc.mean():.4f}')
print(f'Fold ROC-AUC range: {results_df.validation_roc_auc.min():.4f}–{results_df.validation_roc_auc.max():.4f}')

RESULTS = ROOT / "results"
RESULTS.mkdir(exist_ok=True)
results_df.to_csv(RESULTS / "random_forest_baseline.csv", index=False)
settings = {"model": "RandomForestClassifier", "parameters": MODEL_PARAMS,
            "features": FEATURES, "threshold": THRESHOLD, "splits": splits,
            "python": platform.python_version(), "pandas": pd.__version__,
            "numpy": np.__version__, "scikit_learn": sklearn.__version__}
(RESULTS / "random_forest_baseline_settings.json").write_text(json.dumps(settings, indent=2) + "\n")
print("Saved metrics and settings in results/.")

,validation_roc_auc,average_precision,illicit_precision,illicit_recall,illicit_f1
split,,,,,
earlier check,0.8765,0.8469,0.9754,0.7675,0.8590
latest holdout,0.9950,0.9855,0.9952,0.9319,0.9625


Mean fold ROC-AUC: 0.9357
Fold ROC-AUC range: 0.8765–0.9950
Saved metrics and settings in results/.
